In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from datasets import load_dataset 

squad = load_dataset('squad_v2', split='train')
squad

In [ ]:
squad[0]

In [2]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125938 sha256=2fea1d3eeb5148e273fe59ed3da8174ba241032d4adef424d0195c5777f1bb6b
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence_transformers


In [ ]:
from sentence_transformers import InputExample 
from tqdm import tqdm 

train = [] 
for row in tqdm(squad):
    train.append(InputExample(
        texts=[row['question'], row['context']]
    ))

In [ ]:
train = train[:10000]

In [ ]:
from sentence_transformers import datasets

batch_size = 64

loader = datasets.NoDuplicatesDataLoader(
    train, batch_size=batch_size
)

In [ ]:
from sentence_transformers import models, SentenceTransformer

bert = models.Transformer('microsoft/mpnet-base')
pooler = models.Pooling(
    bert.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(modules=[bert, pooler])
model

In [ ]:
from sentence_transformers import losses

loss = losses.MultipleNegativesRankingLoss(model)

In [ ]:
epochs = 1
warmup_steps = int(len(loader) * epochs * 0.1)

# model.fit(
#     train_objectives=[(loader, loss)],
#     epochs=epochs,
#     warmup_steps=warmup_steps,
#     output_path='model1',
#     show_progress_bar=True
# )

In [ ]:
squad_dev = load_dataset('squad_v2', split='validation')
squad_dev[0]

In [ ]:
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')
squad_df = pd.DataFrame() 
for row in tqdm(squad_dev):
    squad_df = squad_df.append({
        'question': row['question'],
        'context': row['context'],
        'id': row['id']
    }, ignore_index=True)

squad_df.head()

In [ ]:
no_dupe = squad_df.drop_duplicates(
    subset='context',
    keep='first'
)

no_dupe = no_dupe.drop(columns=['question'])
no_dupe['id'] = no_dupe['id'] + 'con'
no_dupe.head()

In [ ]:
squad_df = squad_df.merge(no_dupe, how='inner', on='context')
squad_df.head()

In [ ]:
ir_queries = {
    row['id_x']: row['question'] for i, row in squad_df.iterrows()
}
ir_queries

In [ ]:
ir_corpus = {
    row['id_y']: row['context'] for i, row in squad_df.iterrows()
}
ir_corpus

In [ ]:
ir_relevant_docs = {key: [] for key in squad_df['id_x'].unique()}
for i, row in squad_df.iterrows():
    ir_relevant_docs[row['id_x']].append(row['id_y'])
ir_relevant_docs = {key: set(values) for key, values in ir_relevant_docs.items()}
ir_relevant_docs

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

ir_eval = InformationRetrievalEvaluator(
    ir_queries, ir_corpus, ir_relevant_docs
)

In [ ]:
ir_eval(model)

### Data Augmentation with BERT

In [3]:
pairs = [
    ('I am sentence A', 'and this is sentence B'),
    ('there are three pairs', 'each pair includes an A and a B sentence'),
    ('random sampling creates more pairs', 'from these three, we create nine')
]

In [4]:
samples = [] 

for a, _ in pairs:
    for _, b in pairs:
        print((a, b))

('I am sentence A', 'and this is sentence B')
('I am sentence A', 'each pair includes an A and a B sentence')
('I am sentence A', 'from these three, we create nine')
('there are three pairs', 'and this is sentence B')
('there are three pairs', 'each pair includes an A and a B sentence')
('there are three pairs', 'from these three, we create nine')
('random sampling creates more pairs', 'and this is sentence B')
('random sampling creates more pairs', 'each pair includes an A and a B sentence')
('random sampling creates more pairs', 'from these three, we create nine')


In [5]:
import datasets

stsb = datasets.load_dataset('glue', 'stsb', split='train')
stsb_dev = datasets.load_dataset('glue', 'stsb', split='validation')
stsb

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Dataset glue downloaded and prepared to /root/.cache/huggingface/datasets/glue/stsb/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad. Subsequent calls will reuse this data.


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 5749
})

In [6]:
max(stsb['label']), min(stsb['label'])

(5.0, 0.0)

In [7]:
stsb = stsb.map(lambda x: {'label': x['label'] / 5.0})

  0%|          | 0/5749 [00:00<?, ?ex/s]

In [8]:
stsb['label']

[1.0,
 0.7599999904632568,
 0.7599999904632568,
 0.5199999809265137,
 0.8500000238418579,
 0.8500000238418579,
 0.10000000149011612,
 0.3199999928474426,
 0.4399999976158142,
 1.0,
 0.8399999737739563,
 0.9199999570846558,
 0.7734000086784363,
 0.9333999752998352,
 0.33340001106262207,
 0.75,
 1.0,
 0.10000000149011612,
 0.7599999904632568,
 1.0,
 0.6399999856948853,
 0.5600000023841858,
 0.9199999570846558,
 0.6000000238418579,
 1.0,
 0.9600000381469727,
 1.0,
 0.8399999737739563,
 0.8399999737739563,
 0.800000011920929,
 0.800000011920929,
 0.9817999601364136,
 0.6000000238418579,
 0.48000001907348633,
 0.8399999737739563,
 0.6800000071525574,
 1.0,
 0.75,
 0.550000011920929,
 1.0,
 0.800000011920929,
 0.7199999690055847,
 0.3199999928474426,
 0.3499999940395355,
 1.0,
 0.20000000298023224,
 0.20000000298023224,
 0.4749999940395355,
 0.7599999904632568,
 0.6399999856948853,
 0.6399999856948853,
 0.8799999952316284,
 0.75,
 0.949999988079071,
 0.6399999856948853,
 0.31119999289512634,

In [9]:
import torch
from sentence_transformers import InputExample 
from torch.utils.data import DataLoader 

train_data = [] 
for row in stsb:
    train_data.append(
        InputExample(
            texts=[row['sentence1'], row['sentence2']],
            label=float(row['label'])
        )
    )
    
batch_size = 16 
loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [10]:
from sentence_transformers.cross_encoder import CrossEncoder 

cross_encoder = CrossEncoder('bert-base-uncased', num_labels=1)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [11]:
num_epochs = 1 
warmup = int(len(loader) * num_epochs * 0.4)

# cross_encoder.fit(
#     train_dataloader=loader,
#     epochs=num_epochs,
#     warmup_steps=warmup, 
#     output_path='model1'
# )

In [12]:
gold = datasets.load_dataset('glue', 'stsb', split='train')

gold = pd.DataFrame({
    'sentence1': gold['sentence1'],
    'sentence2': gold['sentence2']
})

In [13]:
from tqdm import tqdm 
import warnings 
warnings.filterwarnings('ignore')

pairs = pd.DataFrame() 

for sentence1 in tqdm(list(set(gold['sentence1']))):
    sampled = gold[gold['sentence1'] != sentence1].sample(5)
    sampled = sampled['sentence2'].tolist() 
    for sentence2 in sampled:
        pairs = pairs.append({
            'sentence1': sentence1,
            'sentence2': sentence2
        }, ignore_index=True)

100%|██████████| 5436/5436 [00:50<00:00, 108.59it/s]


In [14]:
len(pairs)
pairs = pairs.drop_duplicates() 
len(pairs)

27178

In [16]:
cross_encoder = CrossEncoder('bert-base-uncased')

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [17]:
silver = list(zip(pairs['sentence1'], pairs['sentence2']))
scores = cross_encoder.predict(silver)

pairs['label'] = scores.tolist()
pairs.head()

Batches:   0%|          | 0/850 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [18]:
all_data = gold.append(pairs, ignore_index=True)

train = []
for _, row in all_data.iterrows():
    train.append(
        InputExample(
            texts=[row['sentence1'], row['sentence2']],
            label=float(row['label'])
        )
    )

loader = DataLoader(train, shuffle=True, batch_size=batch_size)

KeyError: 'label'

In [19]:
from sentence_transformers import models, SentenceTransformer 

bert = models.Transformer('bert-base-uncased')
pooler = models.Pooling(
    bert.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)
model = SentenceTransformer(modules=[bert, pooler])

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [20]:
from sentence_transformers import losses 

loss = losses.CosineSimilarityLoss(model=model)

In [ ]:
epochs = 1
warmup_steps = int(len(loader) * epochs * 0.15)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path='model2'
)